# ASL sign detector: STAC self-training with knowledge distillation

Trains the YOLO11s-P2 detector on the labeled ASL images plus pseudo-labeled unlabeled
images (`training/detector/stac_kd.py`):

1. A **teacher** is trained on the labeled set.
2. Each cycle, the teacher pseudo-labels the unlabeled images (confidence threshold
   0.35 → 0.45 → 0.55) and a fresh **student** is trained on labeled + pseudo-labeled
   images, distilling the teacher's class scores and box distributions.
3. The student becomes the next teacher.

All models are validated on the labeled val split. **Before you start:**
*Runtime → Change runtime type → T4 GPU* (or better), then *Runtime → Run all*.

## 1. Get the project files

If the repository is private, clone with a token (`https://<token>@github.com/...`).

In [ ]:
import os, sys

REPO_URL = "https://github.com/duwcston/VSL-Detection.git"
REPO_BRANCH = "ASL"
REPO_DIR = "/content/VSL-Detection"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
!pip install -q -r {REPO_DIR}/training/detector/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/training/detector")

## 2. Dataset

Downloads the Kaggle dataset (`labeled/` in Ultralytics format with train/val splits,
`unlabeled/` with raw images). Kaggle may ask for credentials if the dataset is private.

In [ ]:
import kagglehub

DATASET_DIR = kagglehub.dataset_download("duwcston/stac-yolo-dataset")
LABELED_YAML = f"{DATASET_DIR}/dataset/labeled/data.yaml"
UNLABELED_DIR = f"{DATASET_DIR}/dataset/unlabeled"
assert os.path.exists(LABELED_YAML) and os.path.isdir(UNLABELED_DIR), os.listdir(DATASET_DIR)
print(open(LABELED_YAML).read())

## 3. Configuration

Keep `USE_DRIVE = True` to write results to Google Drive: if Colab disconnects, run all
cells again and finished models / pseudo-label sets are reused instead of retrained.

Ablations for the paper, one change at a time:
- `kd_weight=0` → STAC self-training without distillation
- `num_cycles=0` → supervised baseline (teacher only)
- `kd_temperature`, `kd_box_weight`, `conf_schedule`

In [ ]:
from stac_kd import Config, STACKD

USE_DRIVE = True
RUN_NAME = "stac_kd"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = f"/content/drive/MyDrive/asl_detector/{RUN_NAME}"
else:
    OUTPUT_ROOT = f"/content/runs/{RUN_NAME}"

cfg = Config(
    labeled_data=LABELED_YAML,
    unlabeled_dir=UNLABELED_DIR,
    output_root=OUTPUT_ROOT,
    num_cycles=3,
    conf_schedule=[0.35, 0.45, 0.55],
    kd_weight=1.0,
    kd_temperature=2.0,
    kd_box_weight=0.5,
    teacher_epochs=150,
    student_epochs=200,
    imgsz=320,  # the backend runs at 320x320
    batch=16,
    patience=30,
    workers=8,
    cache="ram",  # set to "false" if the runtime runs out of RAM
)
cfg

## 4. Train

In [ ]:
pipeline = STACKD(cfg)
final_model = pipeline.run()
print("Final model:", final_model)

## 5. Results

In [ ]:
import json
from IPython.display import Image, display

history = json.load(open(f"{OUTPUT_ROOT}/history.json"))
for name, rec in history.items():
    if name.startswith(("teacher_cycle", "student_cycle")):
        print(f"{name:18s} mAP50 {rec['mAP50']:.4f}  mAP50-95 {rec['mAP50-95']:.4f}  P {rec['precision']:.3f}  R {rec['recall']:.3f}")
    elif name.startswith("pseudo_cycle"):
        print(f"{name:18s} {rec['images']}/{rec['unlabeled_total']} images, {rec['boxes']} boxes @ conf {rec['conf_threshold']}")
print("final:", history["final"])
display(Image(f"{OUTPUT_ROOT}/stac_kd_summary.png"))

## 6. Download

The zip contains `best.pt`, `best.onnx` and `history.json`. To use the model in the app,
copy `best.onnx` to `backend/models/` and point `DEFAULT_MODEL_PATH` in
`backend/app/config/config.py` at it.

In [ ]:
import shutil, tempfile
from google.colab import files

tmp = tempfile.mkdtemp()
for f in ("best.pt", "best.onnx", "history.json", "stac_kd_summary.png"):
    if os.path.exists(f"{OUTPUT_ROOT}/{f}"):
        shutil.copy(f"{OUTPUT_ROOT}/{f}", tmp)
files.download(shutil.make_archive(f"/content/{RUN_NAME}", "zip", tmp))